In [28]:
import sympy as sp
from sympy.physics.vector import dynamicsymbols
from sympy import Eq

# System input components
omega0, omega1, omega2, omega3 = dynamicsymbols('Omega0 Omega1 Omega2 Omega3')  # speeds of each rotor BODY FRAME
omega = sp.Matrix([omega0, omega1, omega2, omega3])
omega

Matrix([
[Omega0(t)],
[Omega1(t)],
[Omega2(t)],
[Omega3(t)]])

In [29]:
# System parameters
d = sp.symbols('d')  # distance from center to each rotor
r0 = sp.Matrix([-d, -d, 0]) / sp.sqrt(2)  # position of rotor 0 in BODY FRAME
r1 = sp.Matrix([d, -d, 0]) / sp.sqrt(2)  # position of rotor 1 in BODY FRAME
r2 = sp.Matrix([d, d, 0]) / sp.sqrt(2)  # position of rotor 2 in BODY FRAME
r3 = sp.Matrix([-d, d, 0]) / sp.sqrt(2)  # position of rotor 3 in BODY FRAME
Cl, Cd = sp.symbols('C_l C_d')  # lift and drag coefficients of each rotor

In [30]:
F_bf = Cl * sp.matrix_multiply_elementwise(omega, omega)
tau_prop = Cd * sp.matrix_multiply_elementwise(omega, omega)  # drag torque produced by each rotor
tau_prop[1] = -tau_prop[1]  # rotor 1 spins opposite direction
tau_prop[3] = -tau_prop[3]  # rotor 3 spins opposite direction
tau_bw = (r0.cross(sp.Matrix([0, 0, F_bf[0]])) +
          r1.cross(sp.Matrix([0, 0, F_bf[1]])) +
          r2.cross(sp.Matrix([0, 0, F_bf[2]])) +
          r3.cross(sp.Matrix([0, 0, F_bf[3]])))
tau_bw[2] += tau_prop[0] + tau_prop[1] + tau_prop[2] + tau_prop[3]
thrust = F_bf[0] + F_bf[1] + F_bf[2] + F_bf[3]
thrust_and_torques = sp.Matrix([thrust, tau_bw[0], tau_bw[1], tau_bw[2]])

In [31]:
# Given desired thrust and torques, solve for required rotor speeds
F_and_tau_des = sp.Matrix(sp.symbols('F_des tau_x_des tau_y_des tau_z_des'))


In [32]:
# Solve for squared omegas first (system is linear in omega^2)
w2_0, w2_1, w2_2, w2_3 = sp.symbols('w2_0 w2_1 w2_2 w2_3')
subs_map = {omega0**2: w2_0, omega1**2: w2_1, omega2**2: w2_2, omega3**2: w2_3}
thrust_and_torques_linear = thrust_and_torques.subs(subs_map)
eq = Eq(thrust_and_torques_linear, F_and_tau_des)
sol_sq = sp.solve(eq, [w2_0, w2_1, w2_2, w2_3], dict=True)[0]

# Recover omegas (assuming positive)
sol_omega = {
    omega0: sp.sqrt(sol_sq[w2_0]),
    omega1: sp.sqrt(sol_sq[w2_1]),
    omega2: sp.sqrt(sol_sq[w2_2]),
    omega3: sp.sqrt(sol_sq[w2_3])
}

sol_omega


{Omega0(t): sqrt(F_des/(4*C_l) - sqrt(2)*tau_x_des/(4*C_l*d) + sqrt(2)*tau_y_des/(4*C_l*d) + tau_z_des/(4*C_d)),
 Omega1(t): sqrt(F_des/(4*C_l) - sqrt(2)*tau_x_des/(4*C_l*d) - sqrt(2)*tau_y_des/(4*C_l*d) - tau_z_des/(4*C_d)),
 Omega2(t): sqrt(F_des/(4*C_l) + sqrt(2)*tau_x_des/(4*C_l*d) - sqrt(2)*tau_y_des/(4*C_l*d) + tau_z_des/(4*C_d)),
 Omega3(t): sqrt(F_des/(4*C_l) + sqrt(2)*tau_x_des/(4*C_l*d) + sqrt(2)*tau_y_des/(4*C_l*d) - tau_z_des/(4*C_d))}